# Image UMAP

**What it does.** Embed single-object crops into two dimensions and plot them with the images themselves.

**When to use it.** For looking at a screen before committing to a classifier — clusters, batch effects and outlier wells show up here first.

**What you get.** A UMAP figure with image thumbnails, and the embedding coordinates.

---

> Every path below is a placeholder. Point `src` at your own data before running.
> Nothing in this notebook writes outside the folder you give it.

## 1. Check the install

If this cell fails, the rest cannot work. It reports the version and whether a GPU is visible — segmentation and training are usable on CPU but slow.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. The function this notebook runs

`spacr.core.generate_image_umap`

```
generate_image_umap(settings=None, return_fig=False)
```

Generate a UMAP or tSNE embedding of per-object features and plot it.

In [ ]:
from spacr.core import generate_image_umap

## 3. Settings

`spacr.settings.set_default_umap_image_settings` fills in every default, so you only have to write down what differs. The cell below prints the full set as it exists in this version — treat that output as the reference, not this notebook.

Change values in `settings`, not in the defaults helper.

In [ ]:
from spacr.settings import set_default_umap_image_settings

defaults = set_default_umap_image_settings({})
for key in sorted(defaults):
    print(f'{key:38s} {defaults[key]!r}')

### Every setting this function accepts

The full dictionary, each key on its own line with its default and what it controls. Edit values in place; delete nothing — a key left at its default behaves exactly as if it were absent.

Generated from this installed version, so it is the real set of keys, the real defaults and the real descriptions.

In [ ]:
settings = {
    # (bool) - After the embedding is clustered, rank every measured
    # feature by how well it separates the clusters - random-forest
    # importance plus a per-feature ANOVA or Kruskal-Wallis test - and
    # write results/cluster_results.csv. Turn it on when you want to
    # know what morphology or intensity feature a cluster actually
    # represents. It adds a full model fit over the whole feature table.
    # Default False.
    'analyze_clusters': False,

    # (str) - Metadata column that identifies independent acquisition
    # batches, normally 'plateID'. Every analyzed row must have a value
    # and at least batch_min_samples rows must occur in each batch. Use
    # an acquisition date or instrument ID only if that is the nuisance
    # source you intend to remove. Default 'plateID'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_column': 'plateID',

    # (bool) - True corrects only the additive batch shift and leaves
    # each batch's scale alone. Use it when the plates differ in level
    # but not in spread, or when a batch has too few rows for a stable
    # variance estimate. False (the default) corrects both location and
    # scale, which is standard ComBat. Ignored by every method other
    # than combat. API: spacr.batch_correction.correct_batch_effects.
    'batch_combat_mean_only': False,

    # (str or None) - Metadata column containing reference-control
    # labels for control_center, normally 'columnID' for plate controls.
    # It is ignored by center, zscore, robust_zscore, and none. Blank
    # follows col_to_compare in Image UMAP or location_column in
    # Classify (ML); regression defaults to 'columnID'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_control_column': None,

    # (str, number, list or None) - Reference/negative-control value(s)
    # in batch_control_column used by control_center. Each plate needs
    # at least batch_min_samples matching rows. Image UMAP falls back to
    # neg and Classify (ML) to negative_control when this field is
    # blank; regression requires an explicit value. Default varies by
    # module. API: spacr.batch_correction.correct_batch_effects.
    'batch_control_values': None,

    # (str) - Optional plate/batch correction applied before Image UMAP,
    # ML screen classification, or phenotype regression. 'none' leaves
    # measurements unchanged; 'center' removes each plate's mean shift;
    # 'zscore' aligns plate means and variances; 'robust_zscore' uses
    # median/MAD and tolerates outliers; 'control_center' estimates only
    # a location shift from reference controls and best preserves
    # treatment dispersion; 'combat' is empirical-Bayes ComBat and needs
    # batch_covariate_column naming the biology to protect, or it
    # refuses to run. Do not correct when plate is confounded with
    # biology. Default 'none'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_correction': 'none',

    # (str, list or None) - Metadata column(s) naming the biology that
    # combat must protect, e.g. 'condition' or 'condition,timepoint'.
    # combat estimates the batch effect from the residuals after these
    # terms, so anything NOT listed here is treated as noise and removed
    # with the plate effect -- leave your treatment out and combat
    # deletes the contrast you are measuring. Required by combat and
    # ignored by every other method; blank makes combat refuse to run
    # rather than silently destroy the signal. Write 'none' to state
    # deliberately that there is no covariate to protect. Default None.
    # API: spacr.batch_correction.correct_batch_effects.
    'batch_covariate_column': None,

    # (int) - Minimum number of rows required in every batch, and
    # minimum matching reference controls per batch for control_center.
    # Correction stops with an actionable error below this threshold
    # because a one- or two-object plate estimate is unstable. Default
    # 3. API: spacr.batch_correction.correct_batch_effects.
    'batch_min_samples': 3,

    # (str) - Policy when control_center cannot find enough reference
    # controls on a plate: 'error' stops rather than silently mixing
    # corrected and raw plates; 'skip' leaves that plate unchanged and
    # records a warning. Default 'error'. API:
    # spacr.batch_correction.correct_batch_effects.
    'batch_missing_control': 'error',

    # (bool) - Choose the standalone/CLI embedding fallback: black
    # canvas with white axes when True, white canvas with black axes
    # when False. In the Qt app, Image UMAP automatically matches its
    # enclosing card in the active theme and uses that theme's readable
    # foreground color instead. Default True.
    'black_background': True,

    # (str) - Algorithm run on the 2D embedding. 'dbscan' grows
    # density-based clusters from eps and min_samples and labels sparse
    # points as noise (-1), discovering the cluster count itself;
    # 'kmeans' (that exact spelling) instead forces exactly min_samples
    # clusters and assigns every point. Choose dbscan for a few distinct
    # phenotypes over a diffuse background, kmeans when you want a fixed
    # number of groups. Default 'dbscan'.
    'clustering': 'dbscan',

    # (str) - Metadata column that identifies the control wells when
    # embedding_by_controls is True: rows whose value equals pos or neg
    # are used to train the reducer, and the column is then dropped
    # before fitting. Typically 'columnID' or 'rowID' depending on where
    # controls sit on the plate. Ignored otherwise. Default 'columnID'.
    'col_to_compare': 'columnID',

    # (str) - Name of a column in the joined measurement table (e.g.
    # 'cond', 'columnID', 'plateID') used to color embedding points
    # instead of the cluster labels. Set it to see how a known grouping
    # such as condition or plate column falls across the map; leave it
    # None to color by the clustering result. Setting it also disables
    # remove_cluster_noise, plot_outlines and smooth_lines. Default
    # None.
    'color_by': None,

    # (str) - Where single-object images come from. 'auto' (the default)
    # uses the pre-generated PNG crop folder when one exists and
    # otherwise cuts each crop out of merged/*.npy on demand; 'png'
    # insists on the folder and fails if it is absent; 'merged' always
    # cuts on demand and ignores the folder even when it is there.
    # On-demand crops are pixel-identical to what the PNG folder would
    # have held for the same settings, so annotations and models stay
    # comparable across the two, and they cost no disk and cannot go
    # stale when crop settings change - at the price of reading the
    # merged array each time. Default 'auto'.
    'crop_source': 'auto',

    # (int) - Matplotlib marker area, in points squared, for each object
    # plotted in the UMAP/tSNE embedding. Increase it when a few hundred
    # points make the scatter look empty; drop it to roughly 5-10 when
    # tens of thousands of points overplot and hide cluster structure.
    # Default 50.
    'dot_size': 50,

    # (bool) - Fit the reducer only on control wells - rows whose
    # col_to_compare value equals pos or neg - and then project every
    # object into that space. Use it when the axes should be defined by
    # the control phenotypes so treatments are read relative to them;
    # False fits on all objects. Default False.
    'embedding_by_controls': False,

    # (float) - DBSCAN neighbourhood radius, expressed in the units of
    # the UMAP/t-SNE embedding and measured with the 'metric' setting:
    # two points are neighbours if they lie within this distance. Raise
    # it to merge fragments into fewer, larger clusters and leave less
    # noise; lower it to split clusters and push more points to noise
    # (-1). Ignored when clustering is 'kmeans'. Default 0.9.
    'eps': 0.9,

    # (str or list) - Names of measurement columns to drop from the
    # feature set before UMAP embedding or ML training, applied after
    # the channel_of_interest selection. Use it to remove features that
    # leak the label or swamp the embedding. It does not filter database
    # rows; use exclude_rows for that. Default None keeps every feature.
    'exclude': None,

    # (list) - Condition labels dropped from the image UMAP input,
    # matched against the cond column that map_condition derives from
    # the pos, neg and mix column IDs; the only possible entries are
    # 'neg', 'pos', 'mix' and 'screen'. A bare string is accepted and
    # wrapped in a list. Use it to embed screen wells only. Default
    # None.
    'exclude_conditions': None,

    # (dict or None) - General UMAP row exclusions. Choose one or more
    # database columns, then check the values whose rows should be
    # removed. Rules are combined with OR, so a row matching any
    # selected column/value pair is excluded. Default None keeps every
    # row.
    'exclude_rows': None,

    # (int) - Base figure size in inches; figures are built square as
    # figuresize x figuresize and font sizes are derived from it
    # (legend, axis labels and ticks at 0.75x, overlay text at 0.5x).
    # Raise it when text is unreadable at publication scale, lower it to
    # fit panels on screen. Default 10; cluster grids cap total width at
    # 200 inches.
    'figuresize': 10,

    # (str or None) - Restricts the feature matrix before dimensionality
    # reduction: only columns matching this channel are kept and the
    # other channel_1-channel_4 columns are dropped. Accepts
    # 'channel_0'-'channel_3', an int, a list of channel numbers, or
    # 'morphology' to keep only shape features (area, eccentricity,
    # Zernike moments, ...). None, 'None', 'all', and '*' disable
    # filtering. Default 'channel_0'.
    'filter_by': 'channel_0',

    # (int) - How many example object crops to draw on the embedding
    # plot: that many per cluster when plot_by_cluster is on (smaller
    # clusters show all they have), otherwise that many sampled at
    # random overall. It also sets how many images each cluster
    # contributes to the cluster-grid figure. Raise it for a fuller
    # montage, lower it when thumbnails hide the points. Default 16.
    'image_nr': 16,

    # (float) - Scale applied to each object thumbnail pasted onto the
    # embedding: 1.0 draws the crop at native pixel size, 0.5 at half.
    # Raise it when crops are too small to judge morphology, lower it
    # when thumbnails overlap and bury the point cloud. Practical range
    # about 0.1-2.0. Default 0.5.
    'img_zoom': 0.5,

    # (bool) - Apply log(x + 1e-6) to every numeric feature, after the
    # correlation filter and before standard scaling. Compresses
    # heavy-tailed measurements such as intensity sums and areas so a
    # handful of bright or huge objects stop dominating the embedding.
    # Negative feature values become NaN and are then filled with the
    # column mean. Default False.
    'log_data': False,

    # (str) - Distance metric used both by the reducer (UMAP or t-SNE)
    # and by DBSCAN clustering, e.g. 'euclidean', 'manhattan', 'cosine'
    # or 'correlation'. Correlation-type metrics compare feature
    # profiles regardless of magnitude and often separate phenotypes
    # better than euclidean on scaled data. Default 'euclidean'.
    'metric': 'euclidean',

    # (float) - UMAP's minimum spacing between points in the 2-D
    # embedding, range 0.0-1.0. Low values (0.0-0.1) let clusters pack
    # tightly and look crisply separated; higher values spread points
    # out and preserve more of the global layout at the cost of visible
    # cluster structure. Ignored when reduction_method is 'tsne'.
    # Default 0.1.
    'min_dist': 0.1,

    # (int) - Meaning depends on 'clustering': for DBSCAN it is how many
    # points must fall within eps for a point to count as a core point,
    # so raising it yields fewer, denser clusters and more noise; for
    # KMeans this same value is reused as n_clusters, the exact number
    # of clusters produced. Lower it (or raise eps) when no clusters are
    # found. Default 100.
    'min_samples': 100,

    # (str) - Plate column ID whose wells hold a mixed positive/negative
    # population; rows with this columnID are labelled cond='mix' for
    # the image UMAP, so they can be coloured separately or dropped via
    # exclude_conditions. Any column matching none of pos, neg or mix is
    # labelled 'screen'. Default 'c3'.
    'mix': 'c3',

    # (int) - CPU workers for parallel stages: measurement, mask
    # adjustment, DataLoader loading, and the sklearn/UMAP calls where
    # -1 means every core. Raise it to shorten CPU-bound steps until RAM
    # or disk I/O saturates. Note the measure-and-crop pipeline
    # overrides your value with cpu_count()-4. Defaults vary by
    # pipeline: cpu_count()-4, -1, or None.
    'n_jobs': -1,

    # (int or float) - Size of the local neighbourhood UMAP balances
    # against global structure, and the perplexity when reduction_method
    # is 'tsne'. Small values (5-50) sharpen fine local structure; large
    # values give a smoother, more global embedding. A float is read as
    # a fraction of the number of objects, and anything below 2 is
    # clamped to 2. Default 1000.
    'n_neighbors': 1000,

    # (str) - Column ID marking negative-control wells in the image
    # UMAP. Rows whose columnID equals it are labelled cond='neg', so
    # exclude_conditions can drop them; and when embedding_by_controls
    # is True the rows whose col_to_compare equals it join pos in
    # fitting the reducer. Default 'c2' (note: not 'c1').
    'neg': 'c2',

    # (float) - Width in points of cluster outlines and interactive
    # selection rings. Smaller values produce thinner boundaries.
    # Default 1.0.
    'outline_width': 1.0,

    # (bool) - Chooses which thumbnails get overlaid on the embedding:
    # when True, up to image_nr crops are sampled from each cluster
    # (DBSCAN noise excluded) so every cluster is represented; when
    # False, image_nr crops are sampled at random across the whole map.
    # Keep True to compare cluster morphologies, False for an unbiased
    # sample. Default True.
    'plot_by_cluster': True,

    # (bool) - Render a second figure with one color-bordered panel per
    # cluster, each filled with up to image_nr example crops from that
    # cluster, and save it as <METHOD>_grid.pdf when save_figure is on.
    # Switch it off to skip the extra render when there are many
    # clusters. Ignored unless plot_images is True. Default True.
    'plot_cluster_grids': True,

    # (bool) - Paste the actual object crops onto the embedding scatter
    # instead of showing bare points. Turn it off for a fast, plain
    # scatter on large datasets - doing so also forces black_background
    # to False and skips the cluster grid figure entirely. Default True.
    'plot_images': True,

    # (bool) - Draw a boundary around each cluster in the embedding - a
    # smoothed hull when smooth_lines is True, otherwise the raw convex
    # hull edges. Helps show cluster extent and overlap but clutters
    # dense maps; clusters with fewer than three points are skipped.
    # Forced off when color_by is set. Default True.
    'plot_outlines': True,

    # (bool) - Show the scatter marker for each object in the embedding.
    # When False the markers are still drawn but at alpha 0, so cluster
    # colors and the legend survive while only the outlines and overlaid
    # thumbnails stay visible - handy for image-only UMAP figures.
    # Marker size comes from dot_size. Default True.
    'plot_points': True,

    # (float) - Opacity of UMAP points from 0 (invisible) to 1 (opaque),
    # used by both static and interactive plots. Default 0.65.
    'point_alpha': 0.65,

    # (str) - Point color for static and interactive UMAP plots. Use
    # 'cluster' or 'viridis' for cluster-based Viridis colors, or any
    # Matplotlib color such as '#4cc9f0', 'orange', or 'white' for one
    # fixed color. Default 'cluster'.
    'point_color': 'cluster',

    # (str) - Column ID marking positive-control wells in the image
    # UMAP. Rows whose columnID equals it are labelled cond='pos', so
    # exclude_conditions can drop them; and when embedding_by_controls
    # is True the rows whose col_to_compare equals it help fit the
    # reducer. Default 'c1' (note: not 'c2').
    'pos': 'c1',

    # (str) - Dimensionality reduction run before clustering and
    # plotting: 'umap' preserves more global structure and can be fitted
    # on controls then applied to all data, 'tsne' emphasises local
    # neighbourhoods and cannot reuse a fitted model. With 'tsne',
    # min_dist is ignored and n_neighbors is used as perplexity.
    # Anything else raises ValueError. Default 'umap'.
    'reduction_method': 'umap',

    # (bool) - Drop the points DBSCAN labelled as noise (-1) from the
    # embedding before it is plotted, so figures show only points that
    # joined a cluster. Turn it off to see everything the reducer
    # placed, including the diffuse background. Meaningless with kmeans,
    # which never emits -1, and it is forced off automatically when
    # color_by is set. Default True.
    'remove_cluster_noise': True,

    # (bool or float) - Before dimensionality reduction, drop numeric
    # features whose absolute Pearson correlation with an already-kept
    # feature exceeds a cut-off. Pass a float to set the cut-off
    # yourself, True to use 0.95, or False to keep everything. Enable it
    # so families of near-duplicate measurements (area, perimeter,
    # convex_area) do not dominate the embedding. Default True.
    'remove_highly_correlated': True,

    # (bool) - When object thumbnails are overlaid on the embedding
    # plot, make zero-valued background pixels fully transparent so only
    # the segmented object shows instead of a black square. Turn it on
    # for a cleaner montage, especially with black_background. Only L, I
    # and RGB crops are supported; other PIL modes raise an error.
    # Default False.
    'remove_image_canvas': False,

    # (bool) - Placeholder for embedding raw crops with ResNet features
    # instead of the measured feature table. The branch in
    # generate_image_umap is an empty pass, so enabling it skips the
    # embedding step entirely and the run then fails on an unbound
    # 'embedding' variable. Leave it False. Default False.
    'resnet_features': False,

    # (int) - Randomly subsample the joined measurement table down to
    # this many objects (fixed seed 42) before dimensionality reduction,
    # keeping UMAP and clustering tractable. Raise it for a more
    # faithful map at higher memory and runtime cost, or set to None to
    # use every row. Must not exceed the available row count. Default
    # 1000.
    'row_limit': 1000,

    # (bool) - Write the embedding, plus the cluster grid when
    # plot_cluster_grids is on, as vector PDFs to
    # <src>/results/<METHOD>_embedding.pdf and <METHOD>_grid.pdf. Enable
    # it when you want the figure for a paper or a record of the run;
    # either way the plots are still displayed on screen. Default False.
    'save_figure': False,

    # (bool) - Draw cluster outlines as a smoothed spline through the
    # convex hull (2 pt wide) rather than the raw straight hull segments
    # (4 pt). Purely cosmetic - it does not change clustering; switch it
    # off if smoothing distorts the true cluster boundary. No effect
    # unless plot_outlines is on, and forced off when color_by is set.
    # Default True.
    'smooth_lines': True,

    # (str, path) - Folder the current step reads from and writes into:
    # raw images for mask generation, the merged/ folder of .npy stacks
    # for measure, the plate root for dataset/regression steps, or the
    # folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/,
    # measurements/measurements.db, datasets/, results/) are created
    # inside it. A list of paths, or a "['a','b']" string, processes
    # several plates in one run.
    'src': 'path',

    # (list) - Measurement tables read from each plate's SQLite database
    # and merged into one analysis DataFrame. Only 'cell', 'nucleus',
    # 'pathogen', 'cytoplasm' and 'png_list' are actually merged, and
    # the image-UMAP step appends 'png_list' itself. Any other name
    # (including 'organelle', which the measure step does write) is
    # loaded and then silently dropped from the merge, and a name
    # missing from the database aborts the read with 'Table not found in
    # database'. List only the compartments your phenotype needs, since
    # each extra table adds its whole feature block. Default ['cell',
    # 'nucleus', 'pathogen', 'cytoplasm'] in most pipelines (['cell']
    # for percent-positive analysis).
    'tables': ['cell', 'cytoplasm', 'nucleus', 'pathogen'],

    # (int) - Initial interactive UMAP chart width in pixels. The
    # chart/sidebar divider can also be dragged while exploring. Default
    # 900.
    'umap_canvas_width': 900,

    # (int) - Initial interactive UMAP image and annotation sidebar
    # width in pixels. The divider remains draggable. Default 280.
    'umap_sidebar_width': 280,

    # (bool) - Print extra run detail instead of the minimal log: the
    # resolved settings table at the start of mask generation, the
    # channel and Cellpose-model choices per object type, per-table row
    # counts and how many objects survive the nuclei/pathogen-per-cell
    # filters when measurement tables are merged, and extra
    # loader/diagnostic output in the training and UMAP paths. It only
    # adds console output, so turn it on when object counts come out
    # unexpected and you need to see which stage removed them. Defaults
    # are per-pipeline: True for mask generation, UMAP, screen analysis,
    # barcode mapping, Cellpose training and plaque analysis; False for
    # measure-and-crop, plot-from-db and plot-from-CSV, the endodyogeny
    # and class-proportion helpers, the Cellpose check/finetune tools,
    # and the screen regression, whose verbose branch display()s the
    # whole per-object score table.
    'verbose': True,

    # (bool) - Whether to visualize the embeddings.
    'visualize': 'cell',

}

# Fill in anything left unset, then check the source path.
settings = set_default_umap_image_settings(settings)
settings['src']

## 4. Run it

This is the long cell. Progress is logged; if you want more of it, raise the log levels in Preferences → Logging, or set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter.

In [ ]:
generate_image_umap(settings)

## Where the output went

A UMAP figure with image thumbnails, and the embedding coordinates.

spaCR writes beside the source folder rather than into a global location, so a plate stays self-contained and re-running does not clobber a different experiment.

### Next steps

* The GUI covers the same workflows with the settings laid out as a form — `python -m spacr`.
* The narrated walkthroughs are at <https://einarolafsson.github.io/spacr/tutorials/>.
* The API reference is at <https://einarolafsson.github.io/spacr/>.